# Step-by-Step Implementation Guide: Bayesian KMRF with Forward Propagation

---

## Part 1: Model Training & Preparation (One-Time Setup)

### **Step 1: Train KMRF Model (What You Already Have)**

This is what you're doing in test_kmrf_new.ipynb:

1. **Initialize KMRF** with your configuration
2. **Run pipeline** to:
   - Get features
   - Load KAMA+MSR labels
   - Prepare training data (up to END_DATE = '2019-01-01')
   - Train XGBoost model
3. **Save the trained model** → This model stays FIXED for entire validation/test period

**Result:** A trained KMRF model that predicts P(regime_{t+1} | features_t)

---

### **Step 2: Extract KAMA+MSR Labels (Historical)**

From your trained KMRF model:

1. **Access the full KAMA+MSR label history** (model.y or similar)
   - These are the "true" regime labels from training period
   - Shape: (n_days_training, 1) with values 0, 1, 2, 3 for original
   
2. **Store these labels** - you'll use them to build the initial HMM transition matrix

**Result:** Historical regime sequence from training period

---

### **Step 3: Estimate Initial HMM Transition Matrix (From Training Data)**

Using ONLY the training period KAMA+MSR labels:

1. **Count regime transitions:**
   - For each consecutive pair of days (t, t+1) in training data
   - Count how many times regime i → regime j occurred
   - Build a 4×4 count matrix (for original classification)

2. **Convert to probabilities:**
   - Divide each row by its sum
   - Add Laplace smoothing (add 1 to all counts) to avoid zeros
   - Result: T[i,j] = P(regime_{t+1} = j | regime_t = i)

3. **Compute steady-state distribution:**
   - Initialize π = [0.25, 0.25, 0.25, 0.25]
   - Iterate: π_new = T^T @ π_old
   - Repeat ~1000 times until convergence
   - Result: π = long-run regime probabilities

**Result:** 
- Transition matrix T (4×4)
- Steady-state π (4,)

---

## Part 2: Daily Prediction Workflow (At Each Rebalancing Day t)

Now you're in the validation or test period. For each rebalancing day t:

### **Step 4: Update KAMA+MSR Labels Through Day t**

This happens BEFORE you predict:

1. **Get all price data up to day t** (not including t+1, t+2, ...)
2. **Recompute KAMA labels** on prices[:t]
   - KAMA smoothing with your parameters
   - Classify as Bull/Bear based on KAMA direction
3. **Recompute MSR labels** on prices[:t]
   - Rolling Sharpe ratio calculations
   - Classify regimes based on mean-return/volatility combinations
4. **Combine KAMA+MSR** to get regime labels for all days up to t

**Key Point:** You're NOT predicting labels for t+1. You're recalculating the historical labels using data available up to t. This is like "refitting the indicators with latest data."

**Result:** Updated regime_labels[:t]

---

### **Step 5: Update HMM Transition Matrix**

Using the updated KAMA+MSR labels from Step 4:

1. **Re-estimate transition matrix** using all data up to day t
   - Same counting process as Step 3
   - But now includes more recent data
   - T_t reflects recent regime transition patterns

2. **Re-compute steady state** from T_t
   - π_t = long-run distribution based on updated transitions

**Why update?** Regime dynamics change over time. Bull markets in 2020 might persist differently than in 2019.

**Result:**
- Updated T_t (4×4)
- Updated π_t (4,)

---

### **Step 6: Get Previous Day's Posterior (Initialize if Day 1)**

You need yesterday's posterior probabilities:

1. **If this is the first rebalancing day:**
   - Initialize: posterior_{t-1} = π_t (steady state)
   - Represents maximum uncertainty (no prior knowledge)

2. **If you've been running for a while:**
   - Use: posterior_{t-1} = the posterior you computed yesterday
   - This creates temporal continuity

**Result:** posterior_{t-1} (4,) - probability distribution over 4 regimes

---

### **Step 7: Compute Features at Day t**

Using ONLY data available up to day t:

1. **Extract price/volume data** through day t
2. **Compute technical indicators:**
   - RSI, MACD, Bollinger Bands, etc.
   - All calculated using data[:t]
3. **Compute cross-asset features:**
   - Correlations with other assets (using recent window)
   - Relative strength indicators
4. **Create feature vector** features_t

**Important:** These are the ACTUAL features at day t, not predicted/forecasted features.

**Result:** features_t (shape depends on your feature set)

---

### **Step 8: Get KMRF Prediction for t+1**

Use your FIXED pre-trained KMRF model:

1. **Feed features_t into KMRF model**
2. **Get prediction:** P(regime_{t+1} | features_t)
   - This is a probability distribution over 4 regimes
   - Shape: (4,)
   - Example: [0.05, 0.15, 0.20, 0.60] → 60% chance of HV Bear at t+1

**Note:** You're only getting ONE prediction (for t+1), not for t+2...t+21.

**Result:** kmrf_likelihood_t (4,) - KMRF's prediction for tomorrow

---

### **Step 9: Compute HMM Prior for t+1**

Using the transition matrix from Step 5 and previous posterior from Step 6:

1. **Matrix multiplication:**
   - prior_t = T_t^T @ posterior_{t-1}
   - This is P(regime_t | regime_{t-1}) marginalized over all possible regime_{t-1}

2. **Interpretation:**
   - "Based on where we were yesterday, where should we expect to be today?"
   - Incorporates regime persistence (regimes tend to continue)

**Result:** prior_t (4,) - HMM's expectation for t+1

---

### **Step 10: Bayesian Fusion (Combine KMRF + HMM)**

Merge the two sources of information:

1. **Element-wise combination:**
   - unnormalized = (prior_t)^(1-α) ⊙ (kmrf_likelihood_t)^α
   - Where ⊙ is element-wise multiplication
   - α is confidence weight (default 0.5)

2. **Normalize to probabilities:**
   - posterior_t = unnormalized / sum(unnormalized)

3. **Interpretation:**
   - When α = 0.5: geometric mean of HMM and KMRF
   - When KMRF and HMM agree: posterior is very confident
   - When they disagree: posterior is more uncertain (flatter distribution)

**Result:** posterior_t (4,) - refined probability for regime at t+1

---

### **Step 11: Forward Propagate for Days t+2 through t+21**

Now propagate the posterior forward without new KMRF predictions:

1. **Initialize forward probabilities array:**
   - forward_probs = zeros(21, 4)
   - forward_probs[0] = posterior_t  # Day t+1 (from Bayesian fusion)

2. **For each day k = 1 to 20 (representing t+2 to t+21):**
   
   a. **HMM one-step transition:**
      - probs_raw = T_t^T @ forward_probs[k-1]
      - This propagates yesterday's distribution forward one day
   
   b. **Apply uncertainty growth:**
      - blend_weight = min(k / 21, 0.5)
      - forward_probs[k] = (1 - blend_weight) × probs_raw + blend_weight × π_t
      - Closer to steady state as we go further out
   
   c. **Store in array**

3. **Result after loop:**
   - forward_probs[0] = P(regime_{t+1})  # From Bayesian fusion
   - forward_probs[1] = P(regime_{t+2})  # Propagated 1 step
   - ...
   - forward_probs[20] = P(regime_{t+21})  # Propagated 20 steps, 50% blended to π

**Result:** forward_probs (21, 4) - probability distributions for next 21 days

---

### **Step 12: Average Forward Probabilities**

For portfolio optimization, you typically want a single distribution:

1. **Average across the 21-day horizon:**
   - avg_probs = mean(forward_probs, axis=0)
   - This gives expected regime probabilities over the rebalancing period

2. **Alternative (time-weighted):**
   - Could weight earlier days more heavily since they're more certain
   - avg_probs = weighted_mean(forward_probs, weights=[exponential decay])

**Result:** avg_probs (4,) - expected regime distribution for portfolio optimization

---

### **Step 13: Save Posterior for Next Day**

Store posterior_t for use in tomorrow's Step 6:

1. **Save to memory/disk:** posterior_{t-1} ← posterior_t
2. **This creates temporal continuity** in your predictions

**Result:** Ready for next rebalancing day

---

## Part 3: Estimating μ and Σ for Portfolio Optimization

Now you have avg_probs = [P(LV_Bull), P(LV_Bear), P(HV_Bull), P(HV_Bear)] for the next 21 days.

### **Step 14: Estimate Regime-Conditional Returns (μ_j)**

Using historical data (training + validation up to day t):

1. **For each regime j ∈ {0, 1, 2, 3}:**
   
   a. **Identify all days in that regime:**
      - days_in_regime_j = dates where KAMA+MSR label = j
      - Use the updated labels from Step 4
   
   b. **Extract returns for those days:**
      - returns_regime_j = daily_returns[days_in_regime_j]
   
   c. **Compute mean return:**
      - μ_j = mean(returns_regime_j)
      - This is the average daily return when in regime j

2. **Result:** μ = [μ_0, μ_1, μ_2, μ_3]

**Example for SPY:**


In [ ]:
μ_0 (LV_Bull):  +0.08% per day (calm bull market)
μ_1 (LV_Bear):  -0.03% per day (calm bear/correction)
μ_2 (HV_Bull):  +0.05% per day (volatile rally)
μ_3 (HV_Bear):  -0.15% per day (panic selling)



---

### **Step 15: Estimate Regime-Conditional Covariance (Σ_j)**

If you have multiple assets (for portfolio), for each regime j:

1. **Extract returns for all assets in regime j:**
   - Returns_regime_j = multi-asset return matrix for days in regime j
   - Shape: (n_days_in_regime_j, n_assets)

2. **Compute covariance matrix:**
   - Σ_j = cov(Returns_regime_j)
   - Shape: (n_assets, n_assets)

3. **Result:** Σ_0, Σ_1, Σ_2, Σ_3 (one covariance matrix per regime)

**For single asset:** Just compute variance σ²_j for each regime.

---

### **Step 16: Compute Unconditional Expected Return**

Combine regime-conditional returns using forward probabilities:

1. **Weighted average:**
   - E[r] = Σ_j avg_probs[j] × μ_j
   - This is the expected return over the next 21 days

2. **Example:**


In [ ]:
avg_probs = [0.40, 0.10, 0.20, 0.30]  # 40% LV Bull, 10% LV Bear, etc.
μ = [0.08%, -0.03%, 0.05%, -0.15%]

E[r] = 0.40×0.08% + 0.10×(-0.03%) + 0.20×0.05% + 0.30×(-0.15%)
     = 0.032% - 0.003% + 0.010% - 0.045%
     = -0.006% per day
     ≈ -0.13% over 21 days



**Result:** E[r] = expected return scalar (or vector if multiple assets)

---

### **Step 17: Compute Unconditional Covariance**

This has two components:

1. **Within-regime variance:**
   - Var_within = Σ_j avg_probs[j] × Σ_j
   - Expected variance if we knew the regime

2. **Between-regime variance:**
   - For each asset i:
     - mean_return_i = E[r_i] (from Step 16)
     - deviation_j = μ_ij - mean_return_i (for each regime j)
     - Var_between = Σ_j avg_probs[j] × (deviation_j)²
   - Uncertainty ABOUT which regime we'll be in

3. **Total variance:**
   - Σ_total = Var_within + Var_between

**For single asset:**


In [ ]:
σ² = Σ_j avg_probs[j] × σ²_j + Σ_j avg_probs[j] × (μ_j - E[r])²



**Result:** Σ (covariance matrix for portfolio) or σ² (variance for single asset)

---

### **Step 18: Scale to 21-Day Horizon**

The returns and variances above are DAILY. Scale to rebalancing period:

1. **Expected return over 21 days:**
   - E[r_21day] = 21 × E[r_daily]
   - Assumes approximately i.i.d. returns (good assumption for short horizons)

2. **Variance over 21 days:**
   - σ²_21day = 21 × σ²_daily
   - Volatility scales with √time

**Result:** 
- E[r_21day] = expected return for next rebalancing period
- σ_21day = volatility for next rebalancing period

---

### **Step 19: Mean-Variance Optimization**

Now you can optimize your portfolio:

1. **Single asset (position sizing):**
   ```
   If E[r_21day] > risk_free_rate:
       optimal_weight = E[r_21day] / (λ × σ²_21day)
   else:
       optimal_weight = 0 (stay in cash)
   ```
   Where λ = risk aversion parameter

2. **Multiple assets:**
   ```
   max_w  w^T × E[r_21day] - (λ/2) × w^T × Σ_21day × w
   s.t.   sum(w) = 1
          w ≥ 0 (if no shorting)
   ```
   Solve using quadratic programming

**Result:** Optimal portfolio weights w

---

### **Step 20: Execute Trades and Move to Next Period**

1. **Rebalance portfolio** according to optimal weights
2. **Wait 21 days** (or your rebalancing period)
3. **On next rebalancing day t+21:**
   - Go back to Step 4
   - Update KAMA+MSR through day t+21
   - Update HMM transition matrix
   - Use posterior_t (from Step 13) as your prior
   - Repeat the entire process

**Result:** Continuous adaptive portfolio management

---

## Summary of Data Flow



In [ ]:
TRAINING (ONE-TIME):
  Historical prices → KAMA+MSR labels → Train KMRF
                                      ↓
                              Save fixed model
                                      ↓
                              Initial transition matrix T

EACH REBALANCING DAY t:
  Prices[:t] → Update KAMA+MSR → Update T_t, π_t
                                       ↓
  Features_t → KMRF (fixed) → P(regime_{t+1})  ← LIKELIHOOD
                                       ↓
  Posterior_{t-1} → T_t^T → P(regime_t | regime_{t-1})  ← PRIOR
                                       ↓
                          BAYESIAN FUSION
                                       ↓
                          Posterior_t (for t+1)
                                       ↓
                    HMM PROPAGATION (21 steps)
                                       ↓
                    Forward_probs[t+1:t+21]
                                       ↓
                          Average → avg_probs
                                       ↓
  Historical regimes → μ_j, Σ_j for each regime
                                       ↓
            avg_probs × μ_j → E[r_21day]
            avg_probs × Σ_j → Σ_21day
                                       ↓
                    PORTFOLIO OPTIMIZATION
                                       ↓
                      Optimal weights w
                                       ↓
                    EXECUTE TRADES



---

## Key Parameters to Set

1. **α (confidence weight):** 0.5 (balance KMRF and HMM)
2. **Uncertainty growth rate:** blend_weight = min(k/21, 0.5)
3. **Rebalancing frequency:** 21 days (monthly)
4. **Laplace smoothing:** Add 1 to transition counts
5. **Risk aversion λ:** Depends on your utility function (typical: 2-5)

---

Ready for me to implement this as code? I'll create:
1. `KMRF_Bayesian` class (extends KMRF)
2. Helper functions for HMM estimation
3. Portfolio optimization utilities
4. Example notebook showing the full workflow